# Bank Marketing ETL Pipeline

## Project Overview

This project demonstrates a simple ETL workflow using Python and Pandas.
The dataset is cleaned, transformed, and separated into three logical tables:

- Client
- Campaign
- Economics

The resulting datasets are exported as CSV files for downstream use.

## Dataset

The dataset contains customer information, campaign interactions, and economic indicators collected during a bank marketing campaign.

Each row represents a client contacted during the campaign.

### Import Libraries

In [8]:
import pandas as pd 
import numpy as np

### Load Dataset

In [9]:
# Load the dataset
marketing = pd.read_csv("../data/raw/bank_marketing_campaign_data.csv") 

# Display the first few rows of the dataset
marketing.head()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no


### Initial Data Exploration

Review selected categorical variables and inspect column data types before applying transformations.

In [ ]:
for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]: 
    print(col) 
    print("-" * 30) 
    print(marketing[col].value_counts()) 
    print() 

# Review the data types of each column
marketing.dtypes

credit_default
------------------------------
credit_default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

mortgage
------------------------------
mortgage
yes        21576
no         18622
unknown      990
Name: count, dtype: int64

previous_outcome
------------------------------
previous_outcome
nonexistent    35563
failure         4252
success         1373
Name: count, dtype: int64

campaign_outcome
------------------------------
campaign_outcome
no     36548
yes     4640
Name: count, dtype: int64



client_id                       int64
age                             int64
job                               str
marital                           str
education                         str
credit_default                    str
mortgage                          str
month                             str
day                             int64
contact_duration                int64
number_contacts                 int64
previous_campaign_contacts      int64
previous_outcome                  str
cons_price_idx                float64
euribor_three_months          float64
campaign_outcome                  str
dtype: object

In [11]:
# Create a mapping of month abbreviations to their corresponding numerical values
month_map = { 
    "jan": 1, 
    "feb": 2, 
    "mar": 3, 
    "apr": 4, 
    "may": 5, 
    "jun": 6, 
    "jul": 7, 
    "aug": 8, 
    "sep": 9, 
    "oct": 10, 
    "nov": 11, 
    "dec": 12 
}

### Create Client Table

This table stores customer demographic and financial information.

| Field | Data Type | Description |
|---------|---------|-------------|
| `client_id` | Integer | Unique identifier assigned to each client. |
| `age` | Integer | Client age in years. |
| `job` | String | Type of occupation reported by the client. |
| `marital` | String | Client marital status. |
| `education` | String | Client education level. |
| `credit_default` | Boolean | Indicates whether the client has credit in default. |
| `mortgage` | Boolean | Indicates whether the client has an active mortgage loan. |

In [ ]:
# Subset the DataFrame to include only the desired columns.
client_columns = ["client_id", "age", "job", "marital", "education", "credit_default", "mortgage"]
client = marketing[client_columns]

# Replace "." with "_".
client["job"] = client["job"].str.replace(".", "_")

# Replace "." with "_" and "unknown" with np.NaN.
client["education"] = (client["education"].str.replace(".", "_").replace("unknown", np.nan))

# Convert to a boolean data type: 1 if "success", otherwise 0.
client["credit_default"] = (client["credit_default"] == "yes").astype(bool)
assert client["credit_default"].dtype == bool
client["mortgage"] = (client["mortgage"] == "yes").astype(bool)
assert client["mortgage"].dtype == bool

# Show DataFrame.
print(client.head())

# Export to CSV.
client.to_csv("client.csv", index = False)

   client_id  age        job  marital    education  credit_default  mortgage
0          0   56  housemaid  married     basic_4y           False     False
1          1   57   services  married  high_school           False     False
2          2   37   services  married  high_school           False      True
3          3   40     admin_  married     basic_6y           False     False
4          4   56   services  married  high_school           False     False


### Create Campaign Table

This table contains information related to marketing campaign interactions.

| Field | Data Type | Description |
|---------|---------|-------------|
| `client_id` | Integer | Unique identifier assigned to each client. |
| `number_contacts` | Integer | Number of contact attempts made during the current marketing campaign. |
| `contact_duration` | Integer | Duration of the most recent contact, measured in seconds. |
| `previous_campaign_contacts` | Integer | Number of contact attempts made during the previous marketing campaign. |
| `previous_outcome` | Boolean | Indicates whether the previous marketing campaign was successful. |
| `campaign_outcome` | Boolean | Indicates whether the current marketing campaign was successful. |
| `last_contact_date` | Date | Date of the most recent client contact. |

In [ ]:
# Subset the DataFrame to include only the desired columns.

campaign_columns = ["client_id", "number_contacts", "contact_duration", "previous_campaign_contacts", "previous_outcome", "campaign_outcome"]
campaign = marketing[campaign_columns]

# Convert to a boolean data type: 1 if "success", otherwise 0.
campaign["previous_outcome"] = (campaign["previous_outcome"] == "success").astype(bool)
assert campaign["previous_outcome"].dtype == bool
campaign["campaign_outcome"] = (campaign["campaign_outcome"] == "yes").astype(bool)
assert campaign["previous_outcome"].dtype == bool

#Create a date column by combining day, month, and a new year column (which should have the value 2022). Format: "YYYY-MM-DD".
campaign["last_contact_date"] = pd.to_datetime({
    "year": 2022,
    "month": marketing.loc[campaign.index, "month"].map(month_map),
    "day": marketing.loc[campaign.index, "day"]
}).dt.strftime("%Y-%m-%d")

# Show DataFrame.
print(campaign.head())

# Export to CSV.
campaign.to_csv("campaign.csv", index = False)

   client_id  number_contacts  contact_duration  previous_campaign_contacts  \
0          0                1               261                           0   
1          1                1               149                           0   
2          2                1               226                           0   
3          3                1               151                           0   
4          4                1               307                           0   

   previous_outcome  campaign_outcome last_contact_date  
0             False             False        2022-05-13  
1             False             False        2022-05-19  
2             False             False        2022-05-23  
3             False             False        2022-05-27  
4             False             False        2022-05-03  


### Create Economics Table

This table stores economic indicators associated with each client record.

| Field | Data Type | Description |
|---------|---------|-------------|
| `client_id` | Integer | Unique identifier assigned to each client. |
| `cons_price_idx` | Float | Consumer Price Index (monthly economic indicator). |
| `euribor_three_months` | Float | Three-month Euribor rate (daily economic indicator). |

In [15]:
# Subset the DataFrame to include only the desired columns.
economics_columns = ["client_id", "cons_price_idx", "euribor_three_months"]
economics = marketing[economics_columns]

# Show DataFrame.
print(economics.head())

# Export to CSV.
economics.to_csv("economics.csv", index = False)

   client_id  cons_price_idx  euribor_three_months
0          0          93.994                 4.857
1          1          93.994                 4.857
2          2          93.994                 4.857
3          3          93.994                 4.857
4          4          93.994                 4.857


### Data Validation

Basic validation checks are performed to verify dataset integrity before export.

In [20]:
# Validate the data.
print("Client table shape:", client.shape)
print("Campaign table shape:", campaign.shape)
print("Economics table shape:", economics.shape)

Client table shape: (41188, 7)
Campaign table shape: (41188, 7)
Economics table shape: (41188, 3)


In [21]:
# Check for duplicates in the client_id column of each table.
assert client["client_id"].is_unique
assert campaign["client_id"].is_unique
assert economics["client_id"].is_unique

In [22]:
# Display summary information about each table.
client.info()
campaign.info()
economics.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   client_id       41188 non-null  int64
 1   age             41188 non-null  int64
 2   job             41188 non-null  str  
 3   marital         41188 non-null  str  
 4   education       39457 non-null  str  
 5   credit_default  41188 non-null  bool 
 6   mortgage        41188 non-null  bool 
dtypes: bool(2), int64(2), str(3)
memory usage: 1.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   client_id                   41188 non-null  int64
 1   number_contacts             41188 non-null  int64
 2   contact_duration            41188 non-null  int64
 3   previous_campaign_contacts  41188 non-null  int64
 4   previous_outcome            4

## Export Results

The transformed datasets are exported as CSV files.

## Conclusion

The original marketing dataset was successfully cleaned, transformed, and separated into three structured tables.

This project demonstrates core ETL tasks, including data cleaning, type standardization, table restructuring, and data export using Python and Pandas.